# 🧩 One universal notebook for many local models — NVIDIA & Apple Silicon

This single notebook replaces the per-model notebooks (`my-mistral-…`, `my-gemma-…`,
`my-gpt-oss-…`, `my-thinkingcap-qwen…`). Pick a model in the **selector cell**, pick a
**precision**, and run top-to-bottom. It: loads an instruction-tuned model from the Hub →
inspects it → tests it 4 ways (pipeline, generate, single tool call, tool conversation) →
saves it to disk → pushes it to your Hub.

## 🖥️ Where it runs

The notebook auto-detects your hardware and adapts. No code changes needed to move between:

| Platform | Backend | 4-bit (NF4) | 16-bit (bf16/fp16) | 32-bit (fp32) |
|:---|:---|:---:|:---:|:---:|
| **RunPod / Linux + NVIDIA** | CUDA | ✅ | ✅ | ✅ |
| **Windows + NVIDIA GPU** | CUDA | ✅ | ✅ | ✅ |
| **macOS (Apple Silicon M-series)** | MPS | ❌ *(no bitsandbytes)* | ✅ | ✅ |
| **Any machine, no GPU** | CPU | ❌ | ⚠️ slow → fp32 | ✅ (slow) |

> **4-bit needs an NVIDIA CUDA GPU.** `bitsandbytes` (the NF4 quantizer) is CUDA-only, so on
> macOS/CPU the 4-bit cell will stop you and ask for **16-bit** or **32-bit** instead. On a Mac,
> 16-bit uses Metal (MPS); a ~7B model in bf16 needs ~16 GB of unified memory, a ~27B needs a
> lot more — prefer smaller models or a beefy Mac.

## 🧠 Pick a model

Choose which model to run in the **"Select the model"** cell (`MODEL_KEY`). Rough footprints
(one loaded copy, add ~10–20% runtime overhead):

| `MODEL_KEY` | Model | Params | 4-bit | 16-bit | 32-bit |
|:---|:---|:---|:---|:---|:---|
| `mistral-7b` | mistralai/Mistral-7B-Instruct-v0.3 | ~7B | ~6 GB | ~16 GB | ~30 GB |
| `gemma-4-e4b` | google/gemma-4-E4B-it | ~4B eff. | ~4 GB | ~9 GB | ~17 GB |
| `gpt-oss-20b` | openai/gpt-oss-20b (MoE) | ~21B | ~13 GB¹ | ~42 GB | ~86 GB |
| `qwen36-27b` | bottlecapai/ThinkingCap-Qwen3.6-27B (reasoning) | ~27B | ~18 GB | ~58 GB | ~112 GB |

¹ gpt-oss ships **natively in MXFP4**; its native kernels only run on Hopper (H100/H200). On any
other GPU it dequantizes to bf16 (~42 GB), and the 4-bit cell here re-quantizes it with NF4.

_RunPod prices/GPUs change often — see [runpod.io/pricing](https://www.runpod.io/pricing).
Set the pod's disk to ≥ 100 GB (≥ 150 GB for the 27B)._

**Tip — free VRAM before loading a bigger model:**
```python
del base_model
import gc, torch; gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
```

# Install libraries

In [ ]:
%pip install torch
%pip install --upgrade transformers
%pip install accelerate peft datasets evaluate trl matplotlib tensorboard sentencepiece

# bitsandbytes powers 4-bit (NF4) quantization but is NVIDIA-CUDA only.
# Install it on Windows/Linux (NVIDIA); skip on macOS (Apple Silicon) / CPU-only machines.
import platform, sys, subprocess
if platform.system() == "Darwin":
    print("macOS detected -> skipping bitsandbytes (4-bit needs an NVIDIA CUDA GPU). Use 16-/32-bit.")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "bitsandbytes"], check=False)

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# 🔍 Detect the hardware

Sets `DEVICE` (`cuda` / `mps` / `cpu`) and two universal helpers used everywhere below:

- **`place(model)`** — puts a model on `DEVICE`. If it was loaded with a `device_map`
  (accelerate already placed it, e.g. 4-bit or multi-GPU), it's left as-is.
- **`RELOAD_KWARGS`** — the `from_pretrained` kwargs for reloading an already-saved model
  (uses `device_map="auto"` on CUDA, nothing on MPS/CPU so `place()` moves it).

In [2]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"   # Apple Silicon (Metal)
else:
    DEVICE = "cpu"

print("Selected DEVICE:", DEVICE)
if DEVICE == "cuda":
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_properties(i).name}")
elif DEVICE == "mps":
    print("  Apple Silicon GPU via Metal (MPS)")
else:
    print("  No GPU found -> running on CPU (slow; prefer 32-bit and a small model)")


def place(model):
    """Move a model to DEVICE unless accelerate already placed it (device_map was used)."""
    return model if getattr(model, "hf_device_map", None) else model.to(DEVICE)


# Kwargs for reloading an already-saved model (the saved config carries the precision).
RELOAD_KWARGS = dict(device_map="auto") if DEVICE == "cuda" else dict()

Selected DEVICE: cuda
  GPU 0: NVIDIA L4


# Login to Hugging Face Hub

In [3]:
import os
from huggingface_hub import login

API_TOKEN = "hf_kBRztpuZTZZMyOKsTYhuXMaGTYwFV"
login(token=API_TOKEN)

HfHubHTTPError: Invalid user token.

In [3]:
import torch

free, total = torch.cuda.mem_get_info()  # bytes, for current device
print(f"Free: {free / 1024**3:.2f} GiB / Total: {total / 1024**3:.2f} GiB")

Free: 21.84 GiB / Total: 22.03 GiB


# 🧠 Select the model

Change **`MODEL_KEY`** to switch models — everything below (loading, on-disk folder name, Hub
repo names, `max_new_tokens`, and a couple of per-model chat-template quirks) is derived from
the entry you pick. Add your own models by extending the `MODELS` dict.

In [4]:
HF_USER = "lukaskellerstein"   # <- your Hub username (used for the repos we push to)

MODELS = {
    "mistral-7b": {
        "model_name": "mistralai/Mistral-7B-Instruct-v0.3",
        "slug": "mistral-7b-v03",
        "max_new_tokens": 256,
        "tool_args_as_dict": False,   # Mistral's template accepts a JSON *string* for arguments
        "reasoning": False,
    },
    "gemma-4-e4b": {
        "model_name": "google/gemma-4-E4B-it",
        "slug": "gemma4-E4B",
        "max_new_tokens": 256,
        "tool_args_as_dict": False,
        "reasoning": False,
    },
    "gpt-oss-20b": {
        "model_name": "openai/gpt-oss-20b",
        "slug": "gpt-oss-20b",
        "max_new_tokens": 512,
        "tool_args_as_dict": False,
        "reasoning": False,
    },
    "qwen36-27b": {
        "model_name": "bottlecapai/ThinkingCap-Qwen3.6-27B",
        "slug": "thinkingcap-qwen36-27B",
        "max_new_tokens": 1024,        # reasoning model -> emits <think>…</think>, needs room
        "tool_args_as_dict": True,     # Qwen's template iterates arguments.items() -> needs a dict
        "reasoning": True,
    },
}

# ▼▼▼ change this line to run a different model ▼▼▼
MODEL_KEY = "gemma-4-e4b"
# ▲▲▲

CFG = MODELS[MODEL_KEY]
MODEL_NAME = CFG["model_name"]
MAX_NEW_TOKENS = CFG["max_new_tokens"]
print(f"Selected model: {MODEL_KEY}  ->  {MODEL_NAME}")
print(f"max_new_tokens={MAX_NEW_TOKENS}  reasoning={CFG['reasoning']}")

Selected model: gemma-4-e4b  ->  google/gemma-4-E4B-it
max_new_tokens=256  reasoning=False


# ⚙️ Choose precision — run ONE of the three cells below

Run **exactly one** of the 4-bit / 16-bit / 32-bit cells. It sets `PRECISION` and `MODEL_KWARGS`,
which the loader below uses. Each cell is **device-aware**:

- **4-bit** requires an NVIDIA CUDA GPU (bitsandbytes) — it will stop you on macOS/CPU.
- **16-bit** uses `bfloat16` on CUDA/MPS (falls back to `float32` on CPU, which can't do half well).
- **32-bit** is `float32` everywhere.

To switch precision later, run a different cell, then re-run the **Load the model** cell.

In [ ]:
# --- 4-bit (bitsandbytes NF4) --- NVIDIA CUDA only ---
import torch
from transformers import BitsAndBytesConfig

assert DEVICE == "cuda", (
    "4-bit (NF4) needs an NVIDIA CUDA GPU (bitsandbytes is CUDA-only). "
    f"Your DEVICE is '{DEVICE}' -> run the 16-bit or 32-bit cell instead."
)

PRECISION = "4bit"
MODEL_KWARGS = dict(
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    ),
    dtype=torch.bfloat16,
    device_map="auto",   # accelerate places the quantized weights
)
print("Selected precision:", PRECISION)

In [5]:
# --- 16-bit (half precision) --- bf16 on CUDA/MPS, fp32 fallback on CPU ---
import torch

PRECISION = "16bit"
if DEVICE == "cpu":
    print("CPU can't do half precision efficiently -> using float32 instead.")
    HALF_DTYPE = torch.float32
else:
    HALF_DTYPE = torch.bfloat16

MODEL_KWARGS = dict(dtype=HALF_DTYPE)
if DEVICE == "cuda":
    MODEL_KWARGS["device_map"] = "auto"   # on MPS/CPU we place() the model after loading
print("Selected precision:", PRECISION, "| dtype:", HALF_DTYPE)

Selected precision: 16bit | dtype: torch.bfloat16


In [5]:
# --- 32-bit (full precision, float32) --- works on every backend ---
import torch

PRECISION = "32bit"
MODEL_KWARGS = dict(dtype=torch.float32)
if DEVICE == "cuda":
    MODEL_KWARGS["device_map"] = "auto"   # on MPS/CPU we place() the model after loading
print("Selected precision:", PRECISION)

Selected precision: 32bit


# Base model from HUB

**What this notebook does:** download the selected instruction-tuned model from the Hugging Face
Hub at the precision you picked, verify it generates a coherent answer, then persist it — save a
copy to local disk **and** push a copy to your own Hub repo.

```mermaid
flowchart LR
    ORIG["🤗 Original Hub<br/>MODEL_NAME"] -->|load| BM["base_model"]
    BM -->|test| OK["✅ generates"]
    BM -->|save| DISK["💾 disk<br/>SAVED_MODEL_&lt;slug&gt;_&lt;prec&gt;"]
    BM -->|push| HUB["📦 Your Hub<br/>…-from-hub"]
    style ORIG fill:#2e7d32,color:#fff,stroke:#1b5e20
```

### Load the model

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Silence transformers' verbose deprecation/config warnings.
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

# Names derived from the selected model + precision -- used by every section below.
SAVED_DIR      = f"SAVED_MODEL_{CFG['slug']}_{PRECISION}"
REPO_FROM_HUB  = f"{HF_USER}/my-base-{CFG['slug']}-{PRECISION}-from-hub"
REPO_FROM_DISK = f"{HF_USER}/my-base-{CFG['slug']}-{PRECISION}-from-disk"

base_model = place(AutoModelForCausalLM.from_pretrained(MODEL_NAME, **MODEL_KWARGS))
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loaded {MODEL_NAME}  |  precision={PRECISION}  |  device={base_model.device}")
print(f"hub repo={REPO_FROM_HUB}")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Loaded google/gemma-4-E4B-it  |  precision=16bit  |  device=cuda:0
hub repo=lukaskellerstein/my-base-gemma4-E4B-16bit-from-hub


In [7]:
import torch

free, total = torch.cuda.mem_get_info()  # bytes, for current device
print(f"Free: {free / 1024**3:.2f} GiB / Total: {total / 1024**3:.2f} GiB")

Free: 7.02 GiB / Total: 22.03 GiB


Log model and tokenizer

In [7]:
def describe(model, tokenizer):
    """Print key facts about a loaded model + tokenizer. Reused in every section below."""
    print("--- Model ---")
    print("Type:", type(model))
    print("Architecture:", model)
    # get_text_config() returns the text sub-config for multimodal models (e.g. Gemma) and
    # the config itself for plain text models, so this line works for every model family.
    print("Vocab size:", model.config.get_text_config().vocab_size)
    print("Input embeddings:", model.get_input_embeddings())
    print("Output embeddings:", model.get_output_embeddings())
    print("--- Tokenizer ---")
    print("Type:", type(tokenizer))
    print("Special tokens:", tokenizer.special_tokens_map)
    print("All tokens count:", len(tokenizer))
    print("Padding side:", tokenizer.padding_side)


describe(base_model, base_tokenizer)

--- Model ---
Type: <class 'transformers.models.gemma4.modeling_gemma4.Gemma4ForConditionalGeneration'>
Architecture: Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (vision_tower): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
      

### Test the model

#### via Pipeline

In [8]:
from transformers import pipeline

PROMPT = "What is AutoGen in abstract?"


def test_pipeline(model, tokenizer, prompt=PROMPT, max_new_tokens=None):
    """Test a model with a text-generation pipeline. Reused in every section below."""
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS
    my_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

    # These are instruction-tuned models, so we pass a chat (list of messages) instead of a raw
    # string. The pipeline auto-applies the tokenizer's chat template for a coherent answer.
    # Reasoning models may emit a <think>…</think> block first -> MAX_NEW_TOKENS is set generously.
    messages = [{"role": "user", "content": prompt}]
    result = my_pipeline(messages, max_new_tokens=max_new_tokens, return_full_text=False)
    print(result[0]["generated_text"])


test_pipeline(base_model, base_tokenizer)

## AutoGen in Abstract: A Framework for Autonomous Agent Collaboration

**AutoGen is an advanced, flexible, and highly configurable framework designed to facilitate the development and deployment of multi-agent conversational systems.** At its core, it moves beyond simple prompt-response interactions by enabling **autonomous agents to interact with each other to solve complex tasks.**

Here is a breakdown of its core concepts in abstract terms:

---

### 1. The Core Concept: Agentic Workflow Orchestration

AutoGen's fundamental abstraction is the **Agent**. Instead of viewing AI as a single monolithic function, AutoGen models it as a system composed of specialized, interacting entities (agents).

* **Agent:** An autonomous entity endowed with specific **roles, capabilities, and conversational logic.** Each agent can be programmed to perform distinct functions (e.g., Coder, Critic, Product Manager, Data Analyst).
* **Conversation/Workflow:** The process by which these agents communicate

#### via Model and Tokenizer

In [9]:
def test_generate(model, tokenizer, prompt=PROMPT, max_new_tokens=None):
    """Test a model by manually applying the chat template + generate. Reused below."""
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS
    # Instruction-tuned model -> build inputs from a chat via the tokenizer's chat template.
    # add_generation_prompt=True appends the assistant marker so the model answers instead of
    # continuing the user's turn.
    messages = [{"role": "user", "content": prompt}]
    model_inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
    ).to(model.device)

    result = model.generate(**model_inputs, max_new_tokens=max_new_tokens)
    print(tokenizer.decode(result[0], skip_special_tokens=True))


test_generate(base_model, base_tokenizer)

user
What is AutoGen in abstract?
model
## AutoGen in Abstract: A Framework for Autonomous Agent Orchestration

**AutoGen is an advanced, multi-agent conversational framework designed to automate complex, multi-step tasks by enabling intelligent agents to interact, collaborate, and self-correct.**

In essence, it moves beyond traditional monolithic AI applications by treating problem-solving not as a single query, but as a **workflow orchestrated by a team of specialized, communicating agents.**

Here is a breakdown of the key abstract concepts:

---

### 1. Multi-Agent Paradigm (The "Team")

The core abstraction is the **Agent**. Instead of a single Large Language Model (LLM) performing all functions, AutoGen defines specialized roles:

* **Specialization:** Agents are not generalists. They can be programmed as a "Coder Agent," a "Product Manager Agent," a "Reviewer Agent," or a "Data Analyst Agent."
* **Heterogeneity:** The system can combine different types of agents (e.g., an LLM-p

#### via Tool call (single turn)

In [10]:
# Single-turn tool calling: pass `tools` to the chat template. An instruction-tuned
# model should answer with a tool call (function name + JSON arguments) instead of prose.
STOCK_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current price of a stock.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The ticker symbol for the stock, e.g. GOOG",
                    }
                },
                "required": ["ticker"],
            },
        },
    }
]


def test_tool_call(model, tokenizer, prompt, tools, max_new_tokens=None):
    """Single-turn tool calling: pass `tools`; the model should emit a tool call. Reused below."""
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS
    messages = [{"role": "user", "content": prompt}]
    model_inputs = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)

    # Greedy decoding -> deterministic tool call. Keep special tokens so the
    # tool-call markers (e.g. [TOOL_CALLS]) stay visible.
    result = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = result[0][model_inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(gen, skip_special_tokens=False))


test_tool_call(base_model, base_tokenizer, "What is the current price of MSFT?", STOCK_TOOLS)

<|tool_call>call:get_stock_price{ticker:<|"|>MSFT<|"|>}<tool_call|><eos>


#### via Conversation with tool calls

In [11]:
import json


def test_tool_conversation(model, tokenizer, max_new_tokens=None):
    """Multi-turn conversation that already contains an assistant tool call + a tool
    response, then a fresh user turn. Confirms the chat template round-trips the
    `tool_calls` / `tool` roles, then answers the latest turn. Reused below.

    NOTE (model-specific): some chat templates iterate the tool-call `arguments` with
    `.items()` (e.g. Qwen3.x) and need a **dict**, while others render a JSON **string**
    (e.g. Mistral). `CFG['tool_args_as_dict']` selects the right form per model."""
    args_obj = {"location": "Paris, France", "format": "celsius"}
    arguments = args_obj if CFG["tool_args_as_dict"] else json.dumps(args_obj)

    conversation = [
        {"role": "user", "content": "What's the weather like in Paris?"},
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {
                    "id": "call12345",
                    "type": "function",
                    "function": {"name": "get_current_weather", "arguments": arguments},
                }
            ],
        },
        {"role": "tool", "content": '{"content": 22}', "tool_call_id": "call12345"},
        {
            "role": "assistant",
            "content": "The current temperature in Paris, France is 22 degrees Celsius.",
        },
        {"role": "user", "content": "What's the weather like in San Francisco right now?"},
    ]

    weather_tools = [
        {
            "type": "function",
            "function": {
                "name": "get_current_weather",
                "description": "Get the current weather",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. San Francisco, CA",
                        },
                        "format": {
                            "type": "string",
                            "enum": ["celsius", "fahrenheit"],
                            "description": "The temperature unit to use. Infer this from the users location.",
                        },
                    },
                    "required": ["location", "format"],
                },
            },
        }
    ]

    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS
    model_inputs = tokenizer.apply_chat_template(
        conversation, tools=weather_tools, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    print("Tokenized conversation input_ids:")
    print(model_inputs["input_ids"])

    # Answer the latest user turn (should be another get_current_weather tool call for SF).
    result = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = result[0][model_inputs["input_ids"].shape[1]:]
    print("\nModel response:")
    print(tokenizer.decode(gen, skip_special_tokens=False))


test_tool_conversation(base_model, base_tokenizer)

Tokenized conversation input_ids:
tensor([[     2,    105,   9731,    107,     46, 163688, 236787,    828, 236779,
           4002, 236779,  19323, 236782,   7777, 236787,     52,   3407,    506,
           1873,   7606,     52, 236764,  19031,  29616,  15921,  29616,   5939,
          29616,   7777, 236787,     52,    818,   4022,   4360,    531,   1161,
         236761, 118936,    672,    699,    506,   5089,   4563, 236761,     52,
         236764,  24626,  24845,     52, 236755,  42024,     52, 236764,     52,
          46435,  15882,     52,   1604,   2084, 236787,     52,  35410,     52,
           1263,   7125,  29616,   7777, 236787,     52,    818,   3207,    532,
           1883, 236764,    545, 236761, 236759, 236761,   5054,  14322, 236764,
           8253,     52, 236764,   2084, 236787,     52,  35410,     52,   5237,
          15979,  24845,     52,   7125,     52, 236764,     52,   5939,     52,
           1604,   2084, 236787,     52,  60688,     52,   1807,     47,   

### Save the model (to disk)

In [ ]:
base_model.save_pretrained(SAVED_DIR)
base_tokenizer.save_pretrained(SAVED_DIR)
print("Saved to", SAVED_DIR)

### Push the model (to HUB)

In [ ]:
# push_to_hub() -> save_pretrained(save_original_format=True) can raise NotImplementedError
# in transformers 5.x, so save with save_original_format=False and upload the folder instead.
from huggingface_hub import upload_folder, create_repo

tmp = f"PUSH_TMP_{CFG['slug']}_{PRECISION}_from_hub"
create_repo(REPO_FROM_HUB, token=API_TOKEN, exist_ok=True)
base_model.save_pretrained(tmp, save_original_format=False)
base_tokenizer.save_pretrained(tmp)
upload_folder(repo_id=REPO_FROM_HUB, folder_path=tmp, token=API_TOKEN)
print("Pushed to", REPO_FROM_HUB)